# Analysis Pipeline — Code Availability

This notebook contains the core analysis code used in the manuscript.

**Sections**
1. Libraries & Imports
2. Data Upload & Preprocessing (Data Quality + Biological Filtering)
3. UMAP Embedding & KMeans Clustering
4. Panel B — Predictive Recovery from Low-Order (≤ Double) Mutants (Parental Excluded)

**Environment**: Google Colab (Python 3). Cells are intended to be executed in order.


## 1. Libraries & Imports

Install pinned dependencies to avoid API/ABI drift, then import all modules
used by downstream sections. A global seed is set for reproducibility.


In [ ]:
# 1. 라이브러리 설치 및 Import
# ==========================================
# [Stability] Pin UMAP / Numba versions to avoid API drift.
# ==========================================
!pip install umap-learn==0.5.11 numba==0.60.0 -q

import os
import io
import random
import warnings
import itertools

import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc

# Machine learning / dimensionality reduction
import umap
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.metrics import silhouette_score

# Colab helpers (file upload / download)
from google.colab import files

warnings.filterwarnings("ignore")

# ==========================================
# [Reproducibility] Fix global random seeds
# ==========================================
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Keep a global list of files that will be offered for download at the end
exported_files = []

print("Libraries loaded and environment initialized.")


## 2. Data Upload & Preprocessing

Reads an uploaded `.csv` / `.xlsx` file with per-replicate measurements, then:

- Filters rows by `Data_Quality` labels (valid + expression-only).
- Applies biological dependency: if expression is missing or the variant is
  labelled *Invalid (Low expression)*, the occupancy (binding) value for that
  variant is set to `NaN` — because binding cannot be reliably inferred from
  a non-expressing variant.
- Aggregates replicates to mean/std per variant (`ID`).
- Computes `log2`-scaled experimental columns (MPNN columns are kept raw).


In [ ]:
# 2. 데이터 업로드 및 전처리 (Data Quality + 생물학적 필터링)

SUCCESS_CRITERION_VALUE = 0.85

# Expression analysis is valid for both groups below, but binding/affinity is
# only trustworthy when expression is sufficient.
VALID_QUALITY_LABELS = ["Valid", "Valid (Non-binding)", "Valid; Valid (Non-binding)"]
EXPRESSION_ONLY_LABELS = ["Invalid (Low expression)"]

print("📂 Upload the analysis data file (.csv or .xlsx).")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file was uploaded.")

filename = next(iter(uploaded))
if filename.endswith(".csv"):
    df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))
else:
    df_raw = pd.read_excel(io.BytesIO(uploaded[filename]))

# --- Defensive column-name cleanup (strip whitespace from Excel headers) ---
df_raw.columns = df_raw.columns.str.strip()

target_id_col = "Mutation_Description"
if target_id_col not in df_raw.columns:
    df_raw.rename(columns={df_raw.columns[0]: target_id_col}, inplace=True)
df_raw[target_id_col] = df_raw[target_id_col].astype(str).str.strip()

# --- Data Quality filtering ---
quality_col = None
for col in df_raw.columns:
    if col.lower() == "data_quality" or ("data" in col.lower() and "quality" in col.lower()):
        quality_col = col
        break

if quality_col:
    print(f"🔍 Quality filter on column: '{quality_col}'")
    df_raw[quality_col] = df_raw[quality_col].astype(str).str.strip()
    total_rows = len(df_raw)
    allowed_labels = VALID_QUALITY_LABELS + EXPRESSION_ONLY_LABELS
    df_raw = df_raw[df_raw[quality_col].isin(allowed_labels)].copy()
    print(f"   {total_rows} → {len(df_raw)} rows retained "
          f"(excluded: {total_rows - len(df_raw)}).")

# --- Numeric coercion of target columns ---
potential_targets = [
    "Normalized_Occupancy",
    "Normalized_Expression",
    "MPNN Z-Score",
    "Relative_MPNN_score",
]
valid_targets = [c for c in potential_targets if c in df_raw.columns]

if not valid_targets:
    raise RuntimeError("No analyzable target columns found in the uploaded file.")

for col in valid_targets:
    df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

# --- Biological-dependency filter on Occupancy (binding) ---
if "Normalized_Occupancy" in valid_targets:
    # (i) Low-expression label → binding invalid
    if quality_col:
        mask_low_exp_label = df_raw[quality_col].isin(EXPRESSION_ONLY_LABELS)
        df_raw.loc[mask_low_exp_label, "Normalized_Occupancy"] = np.nan
    # (ii) Missing expression value → binding invalid
    if "Normalized_Expression" in valid_targets:
        mask_no_exp_val = df_raw["Normalized_Expression"].isna()
        df_raw.loc[mask_no_exp_val, "Normalized_Occupancy"] = np.nan
    invalid_aff_count = df_raw["Normalized_Occupancy"].isna().sum()
    print(f"🧬 Biological filter: {invalid_aff_count} occupancy values "
          f"set to NaN due to expression issues.")

# --- Aggregate replicates to per-variant mean / std ---
df_agg = df_raw.groupby(target_id_col)[valid_targets].agg(["mean", "std"])
df_agg.columns = [f"{a}_{b}" for a, b in df_agg.columns]
df_processed = df_agg.reset_index().rename(columns={target_id_col: "ID"})

# Prefer pre-computed per-variant std columns in the source file, if present
existing_std_cols = {}
for t in valid_targets:
    for candidate in [f"{t}_Std", f"{t}_std", f"Std_{t}"]:
        if candidate in df_raw.columns:
            existing_std_cols[t] = candidate
            break

if existing_std_cols:
    std_only_df = df_raw.groupby(target_id_col)[list(existing_std_cols.values())].mean().reset_index()
    df_processed = pd.merge(df_processed, std_only_df,
                            left_on="ID", right_on=target_id_col, how="left")
    for target, std_col_name in existing_std_cols.items():
        calc_std_col = f"{target}_std"
        if calc_std_col in df_processed.columns:
            df_processed[calc_std_col] = df_processed[calc_std_col].fillna(df_processed[std_col_name])
        else:
            df_processed[calc_std_col] = df_processed[std_col_name]

# --- Create log2 columns for experimental targets (NaNs preserved) ---
for col in valid_targets:
    mean_col = f"{col}_mean"
    log_col = f"Log_{col}"
    if mean_col in df_processed.columns:
        if "MPNN" in col or "Z-Score" in col:
            # MPNN-type scores are kept raw
            df_processed[col] = df_processed[mean_col]
        else:
            df_processed[log_col] = np.log2(df_processed[mean_col].clip(lower=0.05))

print(f"Preprocessing complete: {len(df_processed)} unique variants.")


## 3. UMAP Embedding & KMeans Clustering

Encodes every combinatorial variant using per-residue physicochemical
properties (Kyte–Doolittle hydropathy, residue volume, isoelectric point),
projects the full design space to 2D with DensMAP-enabled UMAP, and
labels points with KMeans (k chosen by silhouette score). Clusters are
finally re-ordered so that `C0 → Ck-1` tracks increasing affinity
(or, when unavailable, expression).


In [ ]:
# 3. UMAP embedding & KMeans clustering (occupancy-prioritized ranking)

# -------- Hyperparameters --------
FINAL_MIN_DIST = 0.1
FINAL_SPREAD = 2.0
N_NEIGHBORS = 50

# Amino-acid features: [Kyte-Doolittle hydropathy, residue volume (Å³), pI]
aa_props = {
    "A": [ 1.8,  88.6, 6.00], "R": [-4.5, 173.4, 10.76], "N": [-3.5, 114.1, 5.41],
    "D": [-3.5, 111.1, 2.77], "C": [ 2.5, 108.5, 5.07], "E": [-3.5, 138.4, 3.22],
    "Q": [-3.5, 143.8, 5.65], "G": [-0.4,  60.1, 5.97], "H": [-3.2, 153.2, 7.59],
    "I": [ 4.5, 166.7, 6.02], "L": [ 3.8, 166.7, 5.98], "K": [-3.9, 168.6, 9.74],
    "M": [ 1.9, 162.9, 5.74], "F": [ 2.8, 189.9, 5.48], "P": [-1.6, 112.7, 6.30],
    "S": [-0.8,  89.0, 5.68], "T": [-0.7, 116.1, 5.60], "W": [-0.9, 227.8, 5.89],
    "Y": [-1.3, 193.6, 5.66], "V": [ 4.2, 140.0, 5.96], "WT":[ 0.0,   0.0, 0.00],
}

# Combinatorial design rules (position: wild-type residue + allowed mutants)
design_rules = {
    52:  {"wt": "T", "muts": ["V"]},
    55:  {"wt": "S", "muts": ["H", "G", "A", "W"]},
    57:  {"wt": "H", "muts": ["Y", "W"]},
    58:  {"wt": "I", "muts": ["H", "D", "T"]},
    99:  {"wt": "V", "muts": ["T"]},
    100: {"wt": "S", "muts": ["K", "T", "A", "L"]},
    103: {"wt": "S", "muts": ["P"]},
    106: {"wt": "S", "muts": ["G"]},
    108: {"wt": "L", "muts": ["S"]},
}

# -------- Build the full combinatorial variant list + feature matrix --------
pos_keys = sorted(design_rules.keys())
all_combos = list(itertools.product(*[
    ["WT"] + [f"{design_rules[p]['wt']}{p}{m}" for m in design_rules[p]["muts"]]
    for p in pos_keys
]))

full_ids, full_vecs = [], []
for combo in all_combos:
    active = [m for m in combo if m != "WT"]
    full_ids.append("Parental" if not active else "_".join(active))
    vec = []
    for i, item in enumerate(combo):
        residue = design_rules[pos_keys[i]]["wt"] if item == "WT" else item[-1]
        vec.extend(aa_props[residue])
    full_vecs.append(vec)

# -------- UMAP projection --------
print("Running UMAP...")
X_full = StandardScaler().fit_transform(np.array(full_vecs))
reducer = umap.UMAP(
    n_neighbors=N_NEIGHBORS,
    min_dist=FINAL_MIN_DIST,
    spread=FINAL_SPREAD,
    densmap=True,
    random_state=RANDOM_STATE,
)
embedding = reducer.fit_transform(X_full)

# -------- Merge UMAP coordinates with preprocessed experimental data --------
if "df_processed" not in globals():
    raise RuntimeError("df_processed is missing; run Section 2 first.")

df_merged_all = pd.DataFrame({
    "ID": full_ids,
    "UMAP1": embedding[:, 0],
    "UMAP2": embedding[:, 1],
}).merge(df_processed, on="ID", how="left")

# -------- KMeans clustering with silhouette-based k selection --------
print("Selecting k by silhouette score...")
umap_coords = df_merged_all[["UMAP1", "UMAP2"]]
best_k, max_sil = 5, -1
for k in range(3, 11):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    score = silhouette_score(umap_coords, km.fit_predict(umap_coords))
    if score > max_sil:
        max_sil, best_k = score, k

kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
df_merged_all["Base_Cluster"] = kmeans.fit_predict(umap_coords)

# -------- Cluster re-ordering (prefer Occupancy → Expression → MPNN) --------
rank_target_col = None
if "Log_Normalized_Occupancy" in df_merged_all.columns:
    rank_target_col = "Log_Normalized_Occupancy"
elif "Log_Normalized_Expression" in df_merged_all.columns:
    rank_target_col = "Log_Normalized_Expression"
else:
    for col in valid_targets:
        log_name = f"Log_{col}"
        if log_name in df_merged_all.columns:
            rank_target_col = log_name
            break
        if col in df_merged_all.columns:
            rank_target_col = col
            break

if rank_target_col is None:
    rank_target_col = "Base_Cluster"

sorted_clusters = sorted(
    range(best_k),
    key=lambda x: df_merged_all.loc[df_merged_all["Base_Cluster"] == x, rank_target_col].mean()
)
reorder_map = {old: new for new, old in enumerate(sorted_clusters)}
df_merged_all["Cluster"] = df_merged_all["Base_Cluster"].map(reorder_map).apply(lambda x: f"C{x}")
ordered_labels = [f"C{i}" for i in range(best_k)]
df_merged_all["Cluster"] = pd.Categorical(df_merged_all["Cluster"],
                                          categories=ordered_labels, ordered=True)

# -------- Save 2D coordinates for downstream plotting --------
out_coords = "1_UMAP_2D_Coordinates.csv"
cols_to_save = ["ID", "UMAP1", "UMAP2", "Cluster"] + [
    c for c in df_merged_all.columns if any(t in c for t in valid_targets)
]
cols_to_save = list(dict.fromkeys(cols_to_save))
df_merged_all[cols_to_save].to_csv(out_coords, index=False)
if out_coords not in exported_files:
    exported_files.append(out_coords)

print(f"UMAP + clustering complete. best_k = {best_k}, "
      f"silhouette = {max_sil:.3f}, rank column = '{rank_target_col}'.")


## 4. Panel B — Predictive Recovery from Low-Order (≤ Double) Mutants

Trains a linear additive (Ridge) model on single + double mutants only
(Parental/WT excluded), predicts scores for the full library, and reports
how much of the *true* global top-1% is recovered as more of the predicted
ranking is screened — separately for Affinity and Expression.

In the manuscript only the **≤ Double** condition is reported, so the
Triple training condition from the exploratory notebook is removed.


In [ ]:
# 4. Panel B: Predictive recovery from low-order (<= Double) mutants
#    Parental / WT excluded from features; single + double mutants used for training.

def plot_low_order_prediction_recovery(
    df,
    aff_col="Log_Normalized_Occupancy",
    exp_col="Log_Normalized_Expression",
    target_top_pct=1,
):
    """Train an additive Ridge model on <= Double mutants and report
    recovery of the global top-`target_top_pct`% as the predicted
    ranking is screened from top to bottom.
    """
    print(f"[Panel B] Predicting global top {target_top_pct}% "
          f"from <= Double mutants (Parental excluded)...\n")

    df_plot = df.copy()

    # ---- 1. Parse the mutation description into a clean list of single mutations ----
    def get_mut_list(m_str):
        if pd.isna(m_str):
            return []
        val = str(m_str).strip().upper()
        if val in ("WT", "NAN", "", "PARENTAL", "NONE"):
            return []
        muts = []
        for m in re.split(r"[,\s_+/]+", str(m_str).strip()):
            m_clean = m.strip()
            if m_clean and m_clean.upper() not in ("PARENTAL", "WT", "NAN", "NONE"):
                muts.append(m_clean)
        return muts

    desc_col = "Mutation_Description" if "Mutation_Description" in df_plot.columns else "ID"
    df_plot["mut_list"]  = df_plot[desc_col].apply(get_mut_list)
    df_plot["mut_count"] = df_plot["mut_list"].apply(len)

    # ---- 2. Multi-label one-hot encoding of pure single-residue features ----
    mlb = MultiLabelBinarizer()
    X_all_sparse = mlb.fit_transform(df_plot["mut_list"])
    feature_names = mlb.classes_
    print(f"  {len(feature_names)} single-mutation features: {list(feature_names)}\n")

    ratios = np.arange(0.01, 1.01, 0.01)
    quantile_val = 1.0 - (target_top_pct / 100.0)

    # Only one training condition is reported in the paper
    train_conditions = {"Trained on ≤ Double": 2}
    color_map        = {"Trained on ≤ Double": "#FF9800"}

    results_list = []

    # ---- 3. Fit & score for each experimental target ----
    for target_col, metric_name in [(aff_col, "Affinity"), (exp_col, "Expression")]:
        if target_col not in df_plot.columns:
            continue

        valid_idx = df_plot[target_col].notna()
        df_valid  = df_plot[valid_idx].copy()
        X_valid   = X_all_sparse[valid_idx]
        y_valid   = df_valid[target_col].values

        # Ground truth global top-X% set
        global_thresh       = df_valid[target_col].quantile(quantile_val)
        actual_top_indices  = set(df_valid[df_valid[target_col] >= global_thresh].index)
        total_hits          = len(actual_top_indices)

        for cond_name, max_muts in train_conditions.items():
            train_mask = df_valid["mut_count"] <= max_muts
            X_train    = X_valid[train_mask]
            y_train    = y_valid[train_mask]

            # L2-regularized linear model (additive approximation)
            model = Ridge(alpha=1.0)
            model.fit(X_train, y_train)

            df_valid["predicted_score"] = model.predict(X_valid)
            df_ranked = df_valid.sort_values("predicted_score", ascending=False)
            total_valid_seqs = len(df_ranked)

            for r in ratios:
                top_n = max(1, int(round(total_valid_seqs * r)))
                selected = set(df_ranked.head(top_n).index)
                recovered = len(selected.intersection(actual_top_indices))
                rec_rate  = (recovered / total_hits * 100) if total_hits > 0 else 0
                results_list.append({
                    "Metric":        metric_name,
                    "Condition":     cond_name,
                    "Ratio":         r * 100,
                    "Recovery_Rate": rec_rate,
                })

    if not results_list:
        print("No data available to plot.")
        return

    df_results = pd.DataFrame(results_list)

    # ---- 4. Plot (one subplot per target) ----
    fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
    sns.set_style("whitegrid")

    metrics = df_results["Metric"].unique()
    for idx, metric in enumerate(metrics):
        ax = axes[idx]
        subset = df_results[df_results["Metric"] == metric]

        for cond_name in train_conditions.keys():
            cond_data = subset[subset["Condition"] == cond_name]
            sns.lineplot(
                data=cond_data, x="Ratio", y="Recovery_Rate", ax=ax,
                linewidth=3, label=cond_name, color=color_map[cond_name],
            )
            ax.fill_between(cond_data["Ratio"], cond_data["Recovery_Rate"],
                            alpha=0.1, color=color_map[cond_name])

            # Mark 10% / 20% reference points
            for r_point in (10.0, 20.0):
                val = cond_data.loc[cond_data["Ratio"] == r_point, "Recovery_Rate"].values[0]
                ax.scatter(r_point, val, color="white",
                           edgecolor=color_map[cond_name],
                           s=60, zorder=5, linewidth=2)
                if r_point == 10.0:
                    ax.text(r_point - 1, val + 4, f"{val:.1f}%",
                            color=color_map[cond_name],
                            fontweight="bold", ha="right")

        ax.set_title(f"{metric} Recovery (Target: Top {target_top_pct}%)",
                     fontsize=15, fontweight="bold", pad=10)
        ax.set_xlabel("Top % of predicted ranking screened",
                      fontsize=12, fontweight="bold")
        if idx == 0:
            ax.set_ylabel(f"Actual top {target_top_pct}% recovery rate (%)",
                          fontsize=12, fontweight="bold")

        ax.set_xlim(0, 100)
        ax.set_ylim(0, 105)
        ax.axvline(10, color="gray", linestyle=":", alpha=0.5)
        ax.axvline(20, color="gray", linestyle=":", alpha=0.5)
        ax.axhline(80, color="gray", linestyle=":", alpha=0.5)
        ax.legend(loc="lower right", fontsize=11, framealpha=0.9)

    plt.suptitle("Predictive power of low-order mutants (additive model)",
                 fontsize=18, fontweight="bold", y=1.05)
    sns.despine()
    plt.tight_layout()

    svg_out   = "PanelB_Predictive_Recovery_LowOrder_Double.svg"
    excel_out = "OriginPro_PanelB_Predictive_Recovery_Double.xlsx"

    plt.savefig(svg_out, format="svg", dpi=600, bbox_inches="tight")
    plt.show()

    df_pivot = (df_results
                .pivot_table(index=["Metric", "Ratio"],
                             columns="Condition",
                             values="Recovery_Rate")
                .reset_index())
    df_pivot.to_excel(excel_out, index=False)
    print(f"Saved figure : {svg_out}")
    print(f"Saved source : {excel_out}")

    try:
        files.download(svg_out)
        files.download(excel_out)
    except Exception:
        pass


# Run against the merged UMAP/experimental dataframe
import re  # re is used inside the function above
if "df_merged_all" in globals():
    plot_low_order_prediction_recovery(df_merged_all, target_top_pct=1)
else:
    print("df_merged_all not found — run Sections 2 and 3 first.")
